# Part 2 - Multi-Objective Clustering (NSGA-II)

This notebook runs and explains the NSGA-II implementation from part2_multiobjective.py.

## 1) Setup

In [ ]:
from pathlib import Path
import os
import sys
import time
from IPython.display import display, Image

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "part2_multiobjective.py").exists():
    PROJECT_ROOT = Path("/home/hafeez/tdm/project_2")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import part2_multiobjective as p2
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
print("NSGA-II configuration:")
print(f"  Population size: {p2.POP_SIZE}")
print(f"  Generations: {p2.N_GENERATIONS}")
print(f"  Mutation rate: {p2.MUTATION_RATE}")
print(f"  Crossover rate: {p2.CROSSOVER_RATE}")
print(f"  k range: [{p2.K_MIN}, {p2.K_MAX}]")

### Objectives and Trade-off

- f1: Intra-cluster compactness (average squared distance to centroid), minimize.
- f2: Number of clusters k, minimize.
- Trade-off: fewer clusters usually increase compactness error, so objectives conflict.

### Plot Helpers

These helpers render figures directly inline so each analysis step can show its chart immediately.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import BytesIO
from IPython.display import Markdown

def show_figure_inline(fig, title):
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=130, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Markdown(f"### {title}"))
    display(Image(data=buf.getvalue()))

## 2) Load Data

In [ ]:
X = p2.load_data()

## 3) Run NSGA-II

In [ ]:
start = time.time()
final_pop, final_fits, hist_f1, hist_f2 = p2.nsga2(X)
print(f"Optimization time: {time.time() - start:.2f} seconds")

### Convergence Plot (After Optimization)

This shows how average Pareto-front compactness and cluster count evolve over generations.

In [ ]:
gens = np.arange(1, len(hist_f1) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(gens, hist_f1, color="#4e79a7", linewidth=2)
axes[0].set_xlabel("Generation")
axes[0].set_ylabel("Avg Compactness (Pareto front)")
axes[0].set_title("Convergence - Compactness (f1)")
axes[0].grid(alpha=0.3)

axes[1].plot(gens, hist_f2, color="#e15759", linewidth=2)
axes[1].set_xlabel("Generation")
axes[1].set_ylabel("Avg k (Pareto front)")
axes[1].set_title("Convergence - Number of Clusters (f2)")
axes[1].grid(alpha=0.3)

show_figure_inline(fig, "NSGA-II Convergence")

## 4) Extract and Inspect Pareto Front

In [ ]:
pareto_pop, pareto_fits = p2.extract_pareto_front(final_pop, final_fits)
print(f"Pareto solution count: {len(pareto_pop)}")
p2.print_pareto_table(pareto_pop, pareto_fits)

### Pareto Front Plot (After Extraction)

This plot shows dominated and non-dominated solutions directly from the current run.

In [ ]:
f1_all = [f[0] for f in final_fits]
f2_all = [f[1] for f in final_fits]
f1_pf = [f[0] for f in pareto_fits]
f2_pf = [f[1] for f in pareto_fits]

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    f2_all, f1_all,
    c="#a8c8e8", alpha=0.5, s=30, edgecolors="none", label="Dominated solutions", zorder=2
)
ax.scatter(
    f2_pf, f1_pf,
    c="#e15759", s=80, edgecolors="#2b2b2b", linewidths=0.7,
    zorder=4, label="Non-dominated (Pareto)"
)
ax.plot(f2_pf, f1_pf, "r--", linewidth=1.5, alpha=0.7, zorder=3)
ax.set_xlabel("Number of Clusters (f2 - minimize)")
ax.set_ylabel("Intra-Cluster Compactness (f1 - minimize)")
ax.set_title("Pareto Front - Multi-Objective Clustering (NSGA-II)")
ax.legend()
ax.grid(alpha=0.3)

show_figure_inline(fig, "Pareto Front")

### Interpretation

- Each point on the Pareto front is a valid trade-off solution.
- Lower k usually increases compactness error, so objectives conflict.
- A practical choice is typically a knee-point balancing both objectives.
- NSGA-II should preserve diversity across different k values.

### Output Verification

In [ ]:
pareto_non_dominated = True
for i, fi in enumerate(pareto_fits):
    for j, fj in enumerate(pareto_fits):
        if i != j and p2.dominates(fj, fi):
            pareto_non_dominated = False
            break
    if not pareto_non_dominated:
        break

all_fit_values = np.array(final_fits, dtype=float)
checks = {
    "population_size_match": len(final_pop) == p2.POP_SIZE,
    "fitness_size_match": len(final_fits) == p2.POP_SIZE,
    "history_length_match": len(hist_f1) == p2.N_GENERATIONS and len(hist_f2) == p2.N_GENERATIONS,
    "pareto_non_empty": len(pareto_fits) > 0,
    "pareto_non_dominated": pareto_non_dominated,
    "all_finite": np.isfinite(all_fit_values).all(),
    "k_bounds_valid": all(p2.K_MIN <= int(ind[0]) <= p2.K_MAX for ind in final_pop),
}

verification_df = pd.DataFrame({"check": list(checks.keys()), "passed": list(checks.values())})
display(verification_df)
if verification_df["passed"].all():
    print("All output checks passed.")
else:
    print("Some checks failed. Inspect the table above.")

## 5) Optional: Save Plot Files

The notebook already shows plots inline; this section is only for exporting PNG files.

In [ ]:
p2.plot_pareto_front(pareto_fits, final_fits)
p2.plot_convergence(hist_f1, hist_f2)

In [ ]:
for name in ["pareto_front.png", "nsga2_convergence.png"]:
    image_path = PROJECT_ROOT / name
    if image_path.exists():
        print(f"Saved: {name} -> {image_path}")
    else:
        print(f"Missing: {name}")